# GIN 직접 짜기: HASH 방식에서 학습하는 GIN 까지

3주차 PracticeCode_3 의 6절 GIN 을 직접 다시 짜요. 1~8번은 브라우저에서 numpy 로 HASH 방식 GIN 층을 만들어 노트북 출력(G1 대 G2 는 구별, C6 대 삼각형 두 개는 못 구별) 과 똑같은지 확인하고, 9~14번은 Colab 에서 torch 로 GINConv 클래스, GIN 모델, 학습 함수를 짜요. 코딩 기초 c01~c12 를 마쳤다고 생각하고 진행해요. 3주차 w3-wl(색 정제), w3-agg(합 집계) 를 먼저 하면 더 쉬워요.

**하는 법**
1. 위 메뉴 **런타임 > 런타임 유형 변경** 은 CPU 그대로 두어도 돼요.
2. 문제마다 **내 코드 칸**을 채우고 실행(Shift+Enter)한 뒤, 바로 아래 **채점 칸**을 실행해요.
3. `통과!` 가 나오면 성공, 빨간 `AssertionError` 가 나오면 마지막 줄의 한국어 안내를 읽고 고쳐요.
4. 막히면 **힌트**를 한 단계씩 펼쳐 보고, 그래도 안 되면 **정답 보기**를 펼쳐요.

## 1. 그래프를 텐서 세 개로 (from_nx)  (따라 치기)

networkx 그래프를 GIN 이 먹는 입력으로 바꿔요: 노드 특징 x, 엣지 목록 edge_index, 정답 y.

- 여기부터 PyTorch 라서 Colab 에서 열려요. 텐서는 c09, 클래스는 c05~c06, nn.Module 은 c10 에서 배웠어요.
- `torch.ones(n, 1)` 은 노드마다 특징 1 하나예요. 노트북 주석 uncolored 는 "색이 없다" 는 뜻이에요.
- `src.extend([u, v])` 는 리스트에 두 개를 한꺼번에 붙여요. 무방향이라 u 에서 v, v 에서 u **두 방향**을 다 넣어요(c08 edge_index).
- `dtype=torch.long` 은 정수 텐서예요. 번호표로 쓰는 edge_index 와 정답은 long 이어야 해요(c11).
- 노트북 원본 첫 줄 `n = G.number_of_nodes()` 도 그대로 쳐요.

아래 코드를 **보면서 직접 쳐 보세요** (복사하지 말고요):

```python
import torch
import networkx as nx
def from_nx(G, y):
    n = G.number_of_nodes()
    x = torch.ones(n, 1)            # 노드마다 특징 1 (색 없음)
    src, dst = [], []
    for u, v in G.edges():
        src.extend([u, v])          # 출발: u, v
        dst.extend([v, u])          # 도착: v, u (두 방향)
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    return x, edge_index, torch.tensor([y])
x, edge_index, y = from_nx(nx.cycle_graph(3), 0)
```

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 networkx 그래프 G 와 정답 번호 y, 출력은 텐서 세 개 x, edge_index, 정답 텐서예요.
2. x 는 노드마다 특징 1 하나라 (노드 수, 1) 모양이에요.
3. edge_index 는 윗줄 출발, 아랫줄 도착인 (2, 엣지 수 x 2) 텐서예요. 무방향이라 엣지 하나를 두 방향으로 넣어요.
4. 손으로: 삼각형 엣지가 (0, 1), (0, 2), (1, 2) 면 src 는 [0, 1, 0, 2, 1, 2], dst 는 [1, 0, 2, 0, 2, 1] 이라 (2, 6) 이에요.
5. 조심할 점: edge_index 는 번호표라 `dtype=torch.long` 이에요. 확인은 `x.shape` 가 (3, 1), `edge_index.shape` 가 (2, 6) 인지 봐요.

슈도코드

```text
노드 수 n 을 센다
x 를 (n, 1) 크기 1 텐서로 만든다
빈 리스트 src, dst 를 만든다
엣지 (u, v) 마다
    src 에 u, v 를 붙인다
    dst 에 v, u 를 붙인다
src 와 dst 를 정수 텐서 edge_index 로 묶는다
x, edge_index, 정답 텐서를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

src 에는 u, v 순서, dst 에는 v, u 순서

</details>

<details><summary>힌트 2</summary>

torch.tensor([src, dst], dtype=torch.long)

</details>

<details><summary>힌트 3</summary>

return x, edge_index, torch.tensor([y])

</details>

원본: PracticeCode_3.ipynb 셀 33 (from_nx)


In [ ]:
# 여기에 코드를 쳐 보세요


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import networkx as nx
assert 'from_nx' in globals() and callable(from_nx), "함수 from_nx 를 만들어야 해요"
assert tuple(x.shape) == (3, 1), f"삼각형의 x 는 (3, 1) 이어야 해요. 지금은 {tuple(x.shape)} 예요"
assert tuple(edge_index.shape) == (2, 6), f"삼각형 엣지 3개를 두 방향으로 넣어 edge_index 는 (2, 6) 이어야 해요. 지금은 {tuple(edge_index.shape)} 예요"
_x, _ei, _y = from_nx(nx.path_graph(4), 1)
assert torch.all(_x == 1) and tuple(_x.shape) == (4, 1), "x 는 torch.ones(n, 1) 이어야 해요"
assert _ei.dtype == torch.long, "edge_index 는 dtype=torch.long 이어야 해요"
assert sorted(zip(_ei[0].tolist(), _ei[1].tolist())) == [(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2)], "edge_index 에 엣지마다 두 방향이 모두 있어야 해요"
assert _y.tolist() == [1], f"y 는 torch.tensor([y]) 여야 해요. 지금은 {_y} 예요"

print("통과! PyG 의 Data(x=..., edge_index=...) 도 같은 두 텐서를 담아요. w3-assign 에서 써요.")

<details><summary>정답 보기</summary>

```python
import torch
import networkx as nx
def from_nx(G, y):
    n = G.number_of_nodes()
    x = torch.ones(n, 1)            # 노드마다 특징 1 (색 없음)
    src, dst = [], []
    for u, v in G.edges():
        src.extend([u, v])          # 출발: u, v
        dst.extend([v, u])          # 도착: v, u (두 방향)
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    return x, edge_index, torch.tensor([y])
x, edge_index, y = from_nx(nx.cycle_graph(3), 0)
```

</details>

## 2. 이웃 합을 index_add_ 로 구하기  (빈칸 채우기)

인접행렬 없이 edge_index 만으로 이웃 특징 합 $\sum_{u} h_u$ 를 구하고, $(1+\varepsilon)h_v$ 를 더해요.

- `torch.zeros_like(x)` 는 x 와 같은 모양의 0 텐서예요. 이웃 합을 담을 빈 칸이에요.
- `src, dst = edge_index` 는 2줄짜리 텐서를 윗줄(출발), 아랫줄(도착) 로 풀어요.
- `x[src]` 는 엣지마다 출발 노드의 특징이에요. `agg.index_add_(0, dst, x[src])` 는 그 값을 **도착 노드 행에 더해** 쌓아요. 0 은 행 방향이라는 뜻이에요.
- 이름 끝 밑줄 `_` 은 "새 텐서를 만들지 않고 agg 자체를 바꾼다" 는 PyTorch 약속이에요.
- 결과는 인접행렬로 한 `A @ x` 와 같아요. 채점에서 둘을 비교해요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 특징 x (4, 1) 과 edge_index (2, 8), 출력은 이웃 합 `agg` (4, 1) 과 `out` (4, 1) 이에요.
2. 인접행렬 없이 엣지를 하나씩 보는 생각이에요. 엣지 (출발 s, 도착 d) 마다 d 행에 x[s] 를 더해요.
3. 손으로: 노드 2 로 들어오는 엣지는 1, 0, 3 에서 와요. 2 + 1 + 4 = 7 이에요. 전체는 [5, 4, 7, 3] 이에요.
4. 조심할 점: `index_add_(0, dst, x[src])` 의 순서예요. 더해지는 자리가 dst, 더하는 값이 x[src] 예요.
5. 확인: `agg.ravel()` 이 [5, 4, 7, 3], `out.ravel()` 이 [6, 6, 10, 7] 이면 맞아요.

슈도코드

```text
x 와 같은 모양 0 텐서 agg 를 만든다
edge_index 를 출발 src 와 도착 dst 로 나눈다
엣지마다
    agg 의 도착 노드 행에 출발 노드 특징을 더한다
out 을 (1 + eps) 곱하기 x 더하기 agg 로 만든다
agg 와 out 을 출력한다
```

</details>

<details><summary>힌트 1</summary>

같은 모양 0 텐서는 zeros_like

</details>

<details><summary>힌트 2</summary>

더해지는 자리는 도착(dst), 더하는 값은 출발 노드 특징 x[src]

</details>

<details><summary>힌트 3</summary>

out = (1.0 + eps) * x + agg

</details>

원본: PracticeCode_3.ipynb 셀 33 (GINConv.forward)


In [ ]:
import torch

# 삼각형 0-1-2 에 꼬리 2-3 (두 방향씩)
edge_index = torch.tensor([[0, 1, 1, 2, 2, 0, 2, 3],
                           [1, 0, 2, 1, 0, 2, 3, 2]])
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])   # 노드 0~3 의 특징
eps = 0.0

agg = torch.___(x)                   # x 와 같은 모양 0 텐서
src, dst = edge_index
agg.index_add_(0, ___, x[___])       # 도착 노드에 출발 노드 특징 더하기
out = (1.0 + eps) * ___ + agg        # 나 + 이웃 합
print(agg.ravel(), out.ravel())

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
assert 'agg' in globals() and 'out' in globals(), "agg 와 out 을 만들어야 해요"
assert tuple(agg.shape) == (4, 1), f"agg 는 x 와 같은 (4, 1) 이어야 해요. 지금은 {tuple(agg.shape)} 예요"
assert torch.allclose(agg.ravel(), torch.tensor([5.0, 4.0, 7.0, 3.0])), f"이웃 합은 [5, 4, 7, 3] 이어야 해요(노드 2 의 이웃은 0, 1, 3). 지금은 {agg.ravel().tolist()} 예요. index_add_(0, dst, x[src]) 순서를 봐요"
_A = torch.zeros(4, 4)
_A[edge_index[1], edge_index[0]] = 1.0
assert torch.allclose(agg, _A @ x), "agg 는 인접행렬로 계산한 A @ x 와 같아야 해요"
assert torch.allclose(out.ravel(), torch.tensor([6.0, 6.0, 10.0, 7.0])), f"out 은 x + 이웃 합 = [6, 6, 10, 7] 이어야 해요. 지금은 {out.ravel().tolist()} 예요"

print("통과! PyG 의 MessagePassing(aggr='add') 도 속에서는 이런 scatter 합을 해요.")

<details><summary>정답 보기</summary>

```python
import torch

# 삼각형 0-1-2 에 꼬리 2-3 (두 방향씩)
edge_index = torch.tensor([[0, 1, 1, 2, 2, 0, 2, 3],
                           [1, 0, 2, 1, 0, 2, 3, 2]])
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])   # 노드 0~3 의 특징
eps = 0.0

agg = torch.zeros_like(x)                   # x 와 같은 모양 0 텐서
src, dst = edge_index
agg.index_add_(0, dst, x[src])       # 도착 노드에 출발 노드 특징 더하기
out = (1.0 + eps) * x + agg        # 나 + 이웃 합
print(agg.ravel(), out.ravel())
```

</details>

## 3. super().__init__() 가 빠진 GINConv  (고치기)

노트북 GINConv 클래스에서 부모 틀 준비 줄이 빠져 객체를 만들 때 에러가 나요. 고쳐요.

- `class GINConv(nn.Module):` 는 PyTorch 가 준 기본 틀 nn.Module 을 물려받아요(c06 상속, c10).
- `__init__` 첫 줄 `super().__init__()` 는 "부모 틀부터 준비" 예요. 빠지면 `cannot assign module before Module.__init__() call` 에러가 나요.
- `self.mlp` 는 Linear, ReLU, Linear 를 차례로 거치는 작은 MLP 예요. 식의 $\mathrm{MLP}$ 가 HASH 표 자리를 대신해요.
- `nn.Parameter` 로 감싼 eps 는 **학습되는 숫자**라 `parameters()` 에 들어가요. 그래서 파라미터는 Linear 두 개의 weight, bias 4개 + eps 1개 = 5개예요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 어디서 막혔는지 나눠요. 에러 마지막 줄은 `cannot assign module before Module.__init__() call` 이에요.
2. 몇 번째 줄인지 봐요. `__init__` 안 `self.mlp = ...` 줄이에요. 부모 틀이 아직 준비 안 됐는데 층을 달려고 했어요.
3. 클래스 설계를 두 칸으로 나눠 봐요. 무엇을 기억하나(속성, `__init__`): 작은 MLP `self.mlp`, 학습되는 숫자 `self.eps`.
4. 무엇을 하나(메서드, `forward`): 이웃 합을 구하고 MLP((1 + eps) x + 이웃 합) 을 돌려줘요. 이 부분은 멀쩡해요.
5. 고치는 법: `__init__` 맨 첫 줄에 `super().__init__()` 을 넣어요. 확인은 `out.shape` 가 (3, 8) 인지 봐요.

슈도코드

```text
GINConv 틀은 nn.Module 을 물려받는다
__init__ 은 in_dim, out_dim, eps_init 을 받는다
    부모 틀을 먼저 준비한다
    Linear, ReLU, Linear 를 이은 self.mlp 를 기억한다
    eps_init 로 시작하는 학습 숫자 self.eps 를 기억한다
forward 는 x 와 edge_index 를 받는다
    이웃 합 agg 를 index_add_ 로 구한다
    self.mlp((1 + self.eps) x + agg) 를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

__init__ 의 맨 첫 줄에 한 줄이 빠졌어요

</details>

<details><summary>힌트 2</summary>

c10 에서 nn.Module 을 물려받을 때 꼭 쓰던 줄

</details>

<details><summary>힌트 3</summary>

super().__init__()

</details>

원본: PracticeCode_3.ipynb 셀 33 (class GINConv)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

class GINConv(nn.Module):
    def __init__(self, in_dim, out_dim, eps_init=0.0):
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.eps = nn.Parameter(torch.tensor([eps_init]))

    def forward(self, x, edge_index):
        n = x.size(0)
        agg = torch.zeros_like(x)
        src, dst = edge_index
        agg.index_add_(0, dst, x[src])          # sum of neighbors
        return self.mlp((1.0 + self.eps) * x + agg)


torch.manual_seed(0)
conv = GINConv(1, 8)
edge_index = torch.tensor([[0, 1, 1, 2, 2, 0], [1, 0, 2, 1, 0, 2]])
out = conv(torch.ones(3, 1), edge_index)
print(out.shape)

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
assert 'GINConv' in globals(), "GINConv 클래스를 만들어야 해요"
assert tuple(out.shape) == (3, 8), f"out 은 (노드 3, 출력 8) 이어야 해요. 지금은 {tuple(out.shape)} 예요"
_c = GINConv(2, 4, eps_init=0.5)
assert len(list(_c.parameters())) == 5, "파라미터는 weight, bias 2쌍 + eps 1개 = 5개여야 해요. eps 를 nn.Parameter 로 감쌌나요?"
assert abs(_c.eps.item() - 0.5) < 1e-6, "eps 는 eps_init 로 시작해야 해요"
_ei = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]])
_x = torch.tensor([[1.0, 0.0], [0.0, 1.0], [2.0, 1.0]])
_A = torch.zeros(3, 3)
_A[_ei[1], _ei[0]] = 1.0
assert torch.allclose(_c(_x, _ei), _c.mlp(1.5 * _x + _A @ _x), atol=1e-5), "forward 결과가 MLP((1 + eps) x + 이웃 합) 과 달라요"

print("통과! 노트북 GIN, Assignment_3 의 GINConv 도 첫 줄이 전부 super().__init__() 예요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

class GINConv(nn.Module):
    def __init__(self, in_dim, out_dim, eps_init=0.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.eps = nn.Parameter(torch.tensor([eps_init]))

    def forward(self, x, edge_index):
        n = x.size(0)
        agg = torch.zeros_like(x)
        src, dst = edge_index
        agg.index_add_(0, dst, x[src])          # sum of neighbors
        return self.mlp((1.0 + self.eps) * x + agg)


torch.manual_seed(0)
conv = GINConv(1, 8)
edge_index = torch.tensor([[0, 1, 1, 2, 2, 0], [1, 0, 2, 1, 0, 2]])
out = conv(torch.ones(3, 1), edge_index)
print(out.shape)
```

</details>

## 4. GIN 모델: 층마다 합해서 이어 붙이기  (빈칸 채우기)

GINConv 를 2층 쌓고, 층마다 노드 합(sum readout) 을 이어 붙여 그래프 벡터를 만드는 GIN 클래스의 빈칸을 채워요.

- `nn.ModuleList([...])` 는 층 여러 개를 리스트로 담되 PyTorch 가 파라미터를 찾을 수 있게 해 줘요. 그냥 리스트면 학습이 안 돼요.
- `GINConv(1 if i == 0 else hidden, hidden)` 은 첫 층만 입력 1칸, 나머지는 hidden 칸이라는 뜻이에요(한 줄 if).
- `t.sum(dim=0)` 은 노드 방향(행) 으로 더해 **그래프 하나에 벡터 하나**를 남겨요. 이게 합 읽기(Sum readout) 예요.
- `torch.cat([...], dim=0)` 으로 층별 합을 이어 붙여요. hidden 16, 2층이면 32칸이에요.
- 모든 노드 특징이 1 이면 C6 과 삼각형 두 개는 어떤 가중치로도 **같은 벡터**가 나와요. 채점에서 확인해요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 무엇을 기억하나(속성, `__init__`): GINConv 층 여러 개 `self.convs`, 마지막 점수 층 `self.readout`.
2. 무엇을 하나(메서드, `forward`): h 를 층마다 통과시키고, 층 결과를 xs 에 모으고, 층마다 노드 합을 이어 붙인 그래프 벡터 g 를 만들어요.
3. 모양을 따라가요. 층 결과 t 는 (노드 수, 16), `t.sum(dim=0)` 은 (16,), 2층을 이어 붙이면 (32,) 예요.
4. 조심할 점: 층 리스트는 `nn.ModuleList` 로 담아야 parameters() 에 잡혀요. 합은 노드 방향 `dim=0` 이에요.
5. 확인: 삼각형으로 `embed=True` 를 부르면 `g.shape` 가 torch.Size([32]) 면 맞아요.

슈도코드

```text
__init__ 에서
    GINConv 층 num_layers 개를 ModuleList 로 기억한다 (첫 층만 입력 1칸)
    hidden x num_layers 칸을 반 점수로 바꾸는 readout 을 기억한다
forward 에서
    빈 리스트 xs 를 만들고 h 를 x 로 둔다
    층 conv 마다
        h 를 conv(h, edge_index) 로 바꾸고 xs 에 붙인다
    xs 의 층마다 노드 방향 합을 구해 이어 붙여 g 로 둔다
    embed 면 g 를, 아니면 readout(g) 를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

층을 담는 리스트는 nn.ModuleList

</details>

<details><summary>힌트 2</summary>

for 안에서는 층 conv 를 함수처럼 불러요: conv(h, edge_index)

</details>

<details><summary>힌트 3</summary>

torch.cat([t.sum(dim=0) for t in xs], dim=0)

</details>

원본: PracticeCode_3.ipynb 셀 33 (class GIN)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

class GINConv(nn.Module):
    def __init__(self, in_dim, out_dim, eps_init=0.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.eps = nn.Parameter(torch.tensor([eps_init]))

    def forward(self, x, edge_index):
        n = x.size(0)
        agg = torch.zeros_like(x)
        src, dst = edge_index
        agg.index_add_(0, dst, x[src])          # sum of neighbors
        return self.mlp((1.0 + self.eps) * x + agg)


class GIN(nn.Module):
    def __init__(self, hidden=16, num_classes=2, num_layers=2):
        super().__init__()
        self.convs = nn.___(
            [GINConv(1 if i == 0 else hidden, hidden) for i in range(num_layers)]
        )
        self.readout = nn.Linear(hidden * num_layers, num_classes)

    def forward(self, x, edge_index, embed=False):
        xs = []
        h = x
        for conv in self.convs:
            h = ___(h, edge_index)
            xs.append(h)
        g = torch.___([t.sum(dim=___) for t in xs], dim=0)  # layer-wise sum pooling
        if embed:
            return g
        return self.readout(g)


torch.manual_seed(0)
model = GIN()
edge_index = torch.tensor([[0, 1, 1, 2, 2, 0], [1, 0, 2, 1, 0, 2]])
g = model(torch.ones(3, 1), edge_index, embed=True)
print(g.shape)

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import networkx as nx
assert 'GIN' in globals(), "GIN 클래스를 만들어야 해요"
assert tuple(g.shape) == (32,), f"그래프 벡터는 hidden 16 x 2층 = (32,) 여야 해요. 지금은 {tuple(g.shape)} 예요. sum(dim=0) 인가요?"
_m = GIN()
assert len(list(_m.parameters())) == 12, "파라미터가 12개(층마다 5개 + readout 2개) 여야 해요. nn.ModuleList 를 썼나요?"
def _t(G):
    s, d = [], []
    for u, v in G.edges():
        s.extend([u, v]); d.extend([v, u])
    return torch.ones(G.number_of_nodes(), 1), torch.tensor([s, d], dtype=torch.long)
assert tuple(_m(*_t(nx.path_graph(5))).shape) == (2,), "embed=False 면 반 2개 점수 (2,) 가 나와야 해요"
with torch.no_grad():
    _a = _m(*_t(nx.cycle_graph(6)), embed=True)
    _b = _m(*_t(nx.disjoint_union(nx.cycle_graph(3), nx.cycle_graph(3))), embed=True)
    _p = _m(*_t(nx.path_graph(5)), embed=True)
    _q = _m(*_t(nx.relabel_nodes(nx.path_graph(5), {0: 4, 1: 3, 2: 2, 3: 1, 4: 0})), embed=True)
assert torch.allclose(_a, _b, atol=1e-4), "C6 과 삼각형 두 개는 같은 그래프 벡터여야 해요(1-WL 한계)"
assert torch.allclose(_p, _q, atol=1e-4), "노드 번호만 바꾼 그래프는 같은 벡터여야 해요. 합 읽기를 노드 방향(dim=0) 으로 했나요?"

print("통과! 노트북 Assignment_3 의 선택 과제 torch GIN 도 이 클래스와 같은 모양이에요(입력 칸만 원자 종류 수).")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

class GINConv(nn.Module):
    def __init__(self, in_dim, out_dim, eps_init=0.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.eps = nn.Parameter(torch.tensor([eps_init]))

    def forward(self, x, edge_index):
        n = x.size(0)
        agg = torch.zeros_like(x)
        src, dst = edge_index
        agg.index_add_(0, dst, x[src])          # sum of neighbors
        return self.mlp((1.0 + self.eps) * x + agg)


class GIN(nn.Module):
    def __init__(self, hidden=16, num_classes=2, num_layers=2):
        super().__init__()
        self.convs = nn.ModuleList(
            [GINConv(1 if i == 0 else hidden, hidden) for i in range(num_layers)]
        )
        self.readout = nn.Linear(hidden * num_layers, num_classes)

    def forward(self, x, edge_index, embed=False):
        xs = []
        h = x
        for conv in self.convs:
            h = conv(h, edge_index)
            xs.append(h)
        g = torch.cat([t.sum(dim=0) for t in xs], dim=0)  # layer-wise sum pooling
        if embed:
            return g
        return self.readout(g)


torch.manual_seed(0)
model = GIN()
edge_index = torch.tensor([[0, 1, 1, 2, 2, 0], [1, 0, 2, 1, 0, 2]])
g = model(torch.ones(3, 1), edge_index, embed=True)
print(g.shape)
```

</details>

## 5. GINConv 의 forward 직접 짜기  (직접 짜기)

__init__ 이 완성된 GINConv 에 forward 를 직접 짜요. $\mathrm{MLP}\big((1+\varepsilon)h_v + \sum_{u} h_u\big)$ 를 그대로 옮기면 돼요.

- forward 는 "데이터가 들어오면 할 일" 이에요(c10). `conv(x, edge_index)` 라고 부르면 forward 가 돌아요.
- 10번에서 한 이웃 합 세 줄 + 12번 식 한 줄이면 끝이에요.
- eps 는 `self.eps`, MLP 는 `self.mlp` 예요. self 는 "지금 이 층 자신" 이에요(c05).
- 채점은 여러 그래프와 eps 값으로 인접행렬 계산 `self.mlp((1 + eps) x + A @ x)` 와 비교해요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 무엇을 기억하나(속성)는 이미 `__init__` 에 있어요. `self.mlp` 와 `self.eps` 두 개예요.
2. 무엇을 하나(메서드, `forward`): 입력 x 는 (노드 수, in_dim), edge_index 는 (2, 엣지 수). 출력은 (노드 수, out_dim) 이에요.
3. 식을 순서로 나눠요. 이웃 합 구하기, (1 + eps) x 더하기, MLP 통과시키기.
4. 이웃 합은 w3-gin-10 의 세 줄 그대로예요. `zeros_like`, src 와 dst 나누기, `index_add_`.
5. 조심할 점: eps 는 숫자 0.0 이 아니라 `self.eps` 를 써야 학습돼요. 마지막 `return` 도 잊지 않아요.
6. 확인: `conv(torch.ones(3, 1), edge_index)` 출력이 3행 4열 텐서면 모양은 맞아요.

슈도코드

```text
forward 는 x 와 edge_index 를 받는다
    x 와 같은 모양 0 텐서 agg 를 만든다
    edge_index 를 src 와 dst 로 나눈다
    agg 의 dst 행마다 x[src] 를 더한다
    (1 + self.eps) 곱하기 x 더하기 agg 를 self.mlp 에 넣은 값을 돌려준다
```

</details>

<details><summary>힌트 1</summary>

w3-gin-10 의 세 줄을 forward 안에 넣어요

</details>

<details><summary>힌트 2</summary>

eps 는 숫자 0.0 이 아니라 self.eps

</details>

<details><summary>힌트 3</summary>

return self.mlp((1.0 + self.eps) * x + agg)

</details>

원본: PracticeCode_3.ipynb 셀 30 (GINConv 식), 셀 33 (forward)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

class GINConv(nn.Module):
    def __init__(self, in_dim, out_dim, eps_init=0.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.eps = nn.Parameter(torch.tensor([eps_init]))

    # forward(self, x, edge_index) 를 완성해요
    # 1. agg = x 와 같은 모양 0 텐서
    # 2. src, dst = edge_index
    # 3. agg.index_add_(0, dst, x[src]) 로 이웃 합
    # 4. self.mlp((1.0 + self.eps) * x + agg) 를 돌려줘요
    def forward(self, x, edge_index):
        ...


torch.manual_seed(0)
conv = GINConv(1, 4)
edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]])
print(conv(torch.ones(3, 1), edge_index))

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import networkx as nx
assert 'GINConv' in globals(), "GINConv 클래스가 있어야 해요"
def _t(G, dim):
    s, d = [], []
    for u, v in G.edges():
        s.extend([u, v]); d.extend([v, u])
    ei = torch.tensor([s, d], dtype=torch.long)
    x = torch.arange(G.number_of_nodes() * dim, dtype=torch.float32).view(-1, dim) / 5.0
    A = torch.zeros(G.number_of_nodes(), G.number_of_nodes())
    A[ei[1], ei[0]] = 1.0
    return x, ei, A
torch.manual_seed(1)
for _G, _dim, _eps in [(nx.path_graph(4), 1, 0.0), (nx.cycle_graph(5), 3, 0.5), (nx.star_graph(4), 2, -0.3)]:
    _c = GINConv(_dim, 6, eps_init=_eps)
    _x, _ei, _A = _t(_G, _dim)
    _got = _c(_x, _ei)
    assert _got is not None, "forward 가 값을 return 해야 해요"
    _want = _c.mlp((1.0 + _eps) * _x + _A @ _x)
    assert tuple(_got.shape) == tuple(_want.shape), f"출력 모양은 {tuple(_want.shape)} 여야 해요. 지금은 {tuple(_got.shape)} 예요"
    assert torch.allclose(_got, _want, atol=1e-5), f"eps={_eps} 일 때 결과가 MLP((1 + eps) x + 이웃 합) 과 달라요. self.eps 와 index_add_(0, dst, x[src]) 를 확인해요"

print("통과! 이 한 층이 GIN 논문(Xu et al., ICLR 2019) 식 그대로예요. PyG 의 GINConv 도 같은 일을 해요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

class GINConv(nn.Module):
    def __init__(self, in_dim, out_dim, eps_init=0.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.eps = nn.Parameter(torch.tensor([eps_init]))

    def forward(self, x, edge_index):
        n = x.size(0)
        agg = torch.zeros_like(x)
        src, dst = edge_index
        agg.index_add_(0, dst, x[src])          # sum of neighbors
        return self.mlp((1.0 + self.eps) * x + agg)


torch.manual_seed(0)
conv = GINConv(1, 4)
edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]])
print(conv(torch.ones(3, 1), edge_index))
```

</details>

## 6. 삼각형 꼬리 대 사각형 꼬리 학습 함수  (직접 짜기)

그래프 목록 전체의 손실을 더해 한 걸음씩 고치는 학습 함수 train_gin 을 짜고, 에폭마다 손실을 리스트로 돌려줘요.

- 노트북은 그래프 16개(노드 6~13개, 삼각형 꼬리는 반 0, 사각형 꼬리는 반 1) 로 학습해요.
- 한 에폭 = 모든 그래프의 `F.cross_entropy(logits.unsqueeze(0), y)` 를 **더한 뒤** 한 번 backward, step 이에요. `unsqueeze(0)` 은 (2,) 를 (1, 2) 로 만들어 "그래프 1개짜리 묶음" 으로 맞춰요.
- 순서는 c11 과 같아요: `model.train()`, `optimizer.zero_grad()`, 손실 계산, `loss.backward()`, `optimizer.step()`.
- 손실 숫자는 `loss.item()` 으로 꺼내 리스트에 담아요.
- 노트북 출력에서도 정확도는 0.50 에 머물고 손실만 13.7 에서 11.1 로 줄어요. 학습률 0.05 가 커서 중간에 손실이 크게 튀었다가 내려와요. 채점은 정확도가 아니라 **손실 값이 원본 학습 반복과 같은지** 봐요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 model, 그래프 목록 graphs (칸마다 x, ei, y), 에폭 수, 학습률. 출력은 에폭마다 손실 숫자 리스트 `losses` 예요.
2. c11 학습 반복 뼈대를 떠올려요. zero_grad, 손실 계산, backward, step. 여기서는 손실 계산 안에 그래프 반복이 하나 더 들어가요.
3. 한 에폭의 손실은 16개 그래프 손실의 합이에요. `loss = 0.0` 에서 시작해 그래프마다 더해요.
4. 조심할 점: backward 와 step 은 그래프 반복이 끝난 뒤 한 번만 해요. logits 는 (2,) 라서 `unsqueeze(0)` 으로 (1, 2) 로 맞춰요.
5. 확인: 에폭 10 번이면 losses 길이가 10, 마지막 값이 첫 값보다 작으면 맞아요.

슈도코드

```text
Adam 옵티마이저 opt 를 만든다
빈 리스트 losses 를 만든다
epochs 번 반복한다
    학습 모드로 두고 기울기를 0 으로 비운다
    loss 를 0 으로 둔다
    그래프 (x, ei, y) 마다
        logits 를 구하고 cross_entropy 손실을 loss 에 더한다
    loss 로 backward 를 하고 opt 로 한 걸음 고친다
    loss 숫자를 losses 에 붙인다
losses 를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

c11-13 학습 함수와 같은 뼈대에 그래프 반복 for 하나가 더 들어가요

</details>

<details><summary>힌트 2</summary>

loss = 0.0 은 에폭마다, 그래프 반복 바깥에서

</details>

<details><summary>힌트 3</summary>

backward 와 step 은 그래프 반복이 끝난 뒤 한 번

</details>

<details><summary>힌트 4</summary>

losses.append(loss.item()) 후 반복이 끝나면 return losses

</details>

원본: PracticeCode_3.ipynb 셀 33 (triangle_tail, square_tail, 학습 반복)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

def from_nx(G, y):
    n = G.number_of_nodes()
    x = torch.ones(n, 1)  # uncolored
    src, dst = [], []
    for u, v in G.edges():
        src.extend([u, v])
        dst.extend([v, u])
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    return x, edge_index, torch.tensor([y])


class GINConv(nn.Module):
    def __init__(self, in_dim, out_dim, eps_init=0.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.eps = nn.Parameter(torch.tensor([eps_init]))

    def forward(self, x, edge_index):
        n = x.size(0)
        agg = torch.zeros_like(x)
        src, dst = edge_index
        agg.index_add_(0, dst, x[src])          # sum of neighbors
        return self.mlp((1.0 + self.eps) * x + agg)


class GIN(nn.Module):
    def __init__(self, hidden=16, num_classes=2, num_layers=2):
        super().__init__()
        self.convs = nn.ModuleList(
            [GINConv(1 if i == 0 else hidden, hidden) for i in range(num_layers)]
        )
        self.readout = nn.Linear(hidden * num_layers, num_classes)

    def forward(self, x, edge_index, embed=False):
        xs = []
        h = x
        for conv in self.convs:
            h = conv(h, edge_index)
            xs.append(h)
        g = torch.cat([t.sum(dim=0) for t in xs], dim=0)  # layer-wise sum pooling
        if embed:
            return g
        return self.readout(g)


def triangle_tail(tail):
    edges = [(0, 1), (1, 2), (2, 0)]
    for i in range(tail):
        edges.append((2 + i, 3 + i))
    return nx.Graph(edges)


def square_tail(tail):
    edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
    for i in range(tail):
        edges.append((3 + i, 4 + i))
    return nx.Graph(edges)


graphs = []
for n in range(6, 14):
    graphs.append(from_nx(triangle_tail(n - 3), 0))  # n nodes
    graphs.append(from_nx(square_tail(n - 4), 1))     # n nodes


# train_gin(model, graphs, epochs, lr) 를 완성해요
# 1. opt = torch.optim.Adam(model.parameters(), lr=lr)
# 2. losses = []
# 3. epochs 번 반복:
#    a. model.train(), opt.zero_grad(), loss = 0.0
#    b. graphs 의 (x, ei, y) 마다 logits = model(x, ei)
#       loss = loss + F.cross_entropy(logits.unsqueeze(0), y)
#    c. loss.backward(), opt.step()
#    d. losses.append(loss.item())
# 4. losses 를 돌려줘요
def train_gin(model, graphs, epochs, lr):
    ...


torch.manual_seed(0)
model = GIN()
losses = train_gin(model, graphs, 10, 0.05)
print(losses[0], losses[-1])

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn.functional as F
assert 'train_gin' in globals() and callable(train_gin), "함수 train_gin 을 만들어야 해요"
torch.manual_seed(0)
_m = GIN()
_L = train_gin(_m, graphs, 10, 0.05)
assert isinstance(_L, list) and len(_L) == 10, f"에폭 10번이면 손실 10개 리스트를 return 해야 해요. 지금은 {_L if not isinstance(_L, list) else len(_L)} 예요"
assert all(isinstance(v, float) for v in _L), "손실은 loss.item() 으로 꺼낸 숫자여야 해요"
torch.manual_seed(0)
_r = GIN()
_o = torch.optim.Adam(_r.parameters(), lr=0.05)
_want = []
for _e in range(10):
    _r.train()
    _o.zero_grad()
    _loss = 0.0
    for _x, _ei, _y in graphs:
        _loss = _loss + F.cross_entropy(_r(_x, _ei).unsqueeze(0), _y)
    _loss.backward()
    _o.step()
    _want.append(_loss.item())
assert abs(_L[0] - _want[0]) < 1e-3, f"첫 에폭 손실이 {_want[0]:.3f} 여야 해요. 지금은 {_L[0]:.3f} 예요. 16개 그래프 손실을 모두 더했나요?"
assert all(abs(a - b) < 1e-3 * max(1.0, abs(b)) for a, b in zip(_L, _want)), f"에폭별 손실이 원본 반복과 달라요. 기대 {[round(v, 3) for v in _want]}, 지금 {[round(v, 3) for v in _L]}. opt.zero_grad() 를 빼먹지 않았나요?"
assert _L[-1] < _L[0], "10 에폭 뒤 손실이 첫 에폭보다 줄어야 해요"

print("통과! Assignment_3 선택 과제도 이 반복으로 MUTAG 학습 그래프를 돌아요. w3-assign 에서 가짜 분자 데이터로 이어서 해요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

def from_nx(G, y):
    n = G.number_of_nodes()
    x = torch.ones(n, 1)  # uncolored
    src, dst = [], []
    for u, v in G.edges():
        src.extend([u, v])
        dst.extend([v, u])
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    return x, edge_index, torch.tensor([y])


class GINConv(nn.Module):
    def __init__(self, in_dim, out_dim, eps_init=0.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.eps = nn.Parameter(torch.tensor([eps_init]))

    def forward(self, x, edge_index):
        n = x.size(0)
        agg = torch.zeros_like(x)
        src, dst = edge_index
        agg.index_add_(0, dst, x[src])          # sum of neighbors
        return self.mlp((1.0 + self.eps) * x + agg)


class GIN(nn.Module):
    def __init__(self, hidden=16, num_classes=2, num_layers=2):
        super().__init__()
        self.convs = nn.ModuleList(
            [GINConv(1 if i == 0 else hidden, hidden) for i in range(num_layers)]
        )
        self.readout = nn.Linear(hidden * num_layers, num_classes)

    def forward(self, x, edge_index, embed=False):
        xs = []
        h = x
        for conv in self.convs:
            h = conv(h, edge_index)
            xs.append(h)
        g = torch.cat([t.sum(dim=0) for t in xs], dim=0)  # layer-wise sum pooling
        if embed:
            return g
        return self.readout(g)


def triangle_tail(tail):
    edges = [(0, 1), (1, 2), (2, 0)]
    for i in range(tail):
        edges.append((2 + i, 3 + i))
    return nx.Graph(edges)


def square_tail(tail):
    edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
    for i in range(tail):
        edges.append((3 + i, 4 + i))
    return nx.Graph(edges)


graphs = []
for n in range(6, 14):
    graphs.append(from_nx(triangle_tail(n - 3), 0))  # n nodes
    graphs.append(from_nx(square_tail(n - 4), 1))     # n nodes


def train_gin(model, graphs, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        loss = 0.0
        for x, ei, y in graphs:
            logits = model(x, ei)
            loss = loss + F.cross_entropy(logits.unsqueeze(0), y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


torch.manual_seed(0)
model = GIN()
losses = train_gin(model, graphs, 10, 0.05)
print(losses[0], losses[-1])
```

</details>